# Vertex AI Feature Store Pipeline
--------------------------------

**Architecture:**
Feature Engineering
        │
        ▼
Vertex Feature Store
        │
Streaming Ingestion
        │
        ▼
Online Feature Retrieval
        │
        ▼
Model Serving

---
**Purpose**:
Demonstrate creation and SDK-driven operation of a Vertex AI Feature Store, including streaming ingestion and online feature retrieval.

- Required Notebooks Before Execution:
- 01_exploration.ipynb  
- 02_feature_engineering.ipynb  
- 03_training_keras_vertex.ipynb  
- 04_vertex_ai_custom_training_job.ipynb  
- 05_deploy_vertex.ipynb  
- 06_course_production_ml_systems.ipynb (Markdown only)

Scope  

- Create Feature Store from scratch  
- Create EntityType  
- Define features  
- Enable online serving  
- Stream feature values via SDK  
- Retrieve online feature values  

This notebook executes real infrastructure and therefore:
- Uses environment-driven configuration
- Avoids console-created resources
- Is designed to be reproducible
- Uses explicit existence checks (no try/except control flow)

## Section 0A — Install Vertex SDK

In [1]:
# IMPORTANT: After this cell runs, you MUST restart your Colab runtime
# (Runtime -> Restart runtime...) and then run all cells from the beginning
# to ensure the new library version is fully loaded and all resources are re-created.

import os

# Install a specific, stable version of the google-cloud-aiplatform library
# This helps prevent unexpected API changes that can cause persistent errors.

#Global -- Recommended
try:
    import google.cloud.aiplatform
    print("google-cloud-aiplatform is already installed.\n")
    !pip show google-cloud-aiplatform
except ImportError:
    print("google-cloud-aiplatform not found. Installing Global now...\n")
    !pip install --upgrade google-cloud-aiplatform
    import google.cloud.aiplatform
    print("google-cloud-aiplatform -- global has been installed.\n")
    !pip show google-cloud-aiplatform


# User -- Use if Global causes errors.
# !pip install google-cloud-aiplatform==1.47.0 --user

# try:
#     import google.cloud.aiplatform
#     print("google-cloud-aiplatform is already installed.")
#     !pip show google-cloud-aiplatform
# except ImportError:
#     print("google-cloud-aiplatform not found. Installing User now...")
#     !pip install google-cloud-aiplatform==1.47.0 --user
#     import google.cloud.aiplatform
#     print("google-cloud-aiplatform -- user has been installed.")
#     !pip show google-cloud-aiplatform

google-cloud-aiplatform is already installed.

Name: google-cloud-aiplatform
Version: 1.140.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, google-genai, packaging, proto-plus, protobuf, pydantic, typing_extensions
Required-by: google-adk


## Section 0B — Secrets

In [2]:
from google.colab import userdata

PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
DATA_PREP_PREFIX = userdata.get("DATA_PREP_PREFIX")
MLOPS_PREFIX = userdata.get("MLOPS_PREFIX")
DEPLOYMENT_PREFIX = userdata.get("DEPLOYMENT_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"

In [3]:
from google.cloud.aiplatform_v1.types import FeatureValue
from datetime import datetime, timezone

## Section 0C - Environment Verification

In [4]:
## Section 0C — Environment Verification

import google.cloud.aiplatform
import sys

print("Environment Verification")
print("------------------------")

print(f"Python Version: {sys.version}")
print(f"Vertex AI SDK Version: {google.cloud.aiplatform.__version__}")

print("\nProject Configuration")
print("---------------------")
print(f"PROJECT_ID: {PROJECT_ID}")
print(f"REGION: {REGION}")
print(f"GCS_BUCKET: {GCS_BUCKET}")

# Safety check
if not PROJECT_ID or not REGION:
    raise ValueError("PROJECT_ID and REGION must be defined in Colab secrets.")

print("\nEnvironment verification complete.")

Environment Verification
------------------------
Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Vertex AI SDK Version: 1.140.0

Project Configuration
---------------------
PROJECT_ID: first-haven-485022-n5
REGION: us-central1
GCS_BUCKET: gs://ml-cert-sandbox-bucket-js-001

Environment verification complete.


## Section 1 — Authenticate & Configure Gcloud

In [5]:
from google.colab import auth
auth.authenticate_user()  # Ensure Colab has GCS / Vertex AI access

# !gcloud auth login --quiet # was missing
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION
!gcloud config list project

# Confirm authentication & configuration
print("Project:")
!gcloud config get-value project
print("\nAuthenticated User Account:")
!gcloud config get-value account
print("\nRegion:")
!gcloud config get-value compute/region

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Updated property [compute/region].
[core]
project = first-haven-485022-n5

Your active configuration is: [default]
Project:
first-haven-485022-n5

Authenticated User Account:
davistyrant@gmail.com

Region:
us-central1


Production Notes:

Always authenticate via auth.authenticate_user() for reproducibility.

Initialization must point to correct project/region to avoid resource collisions.

## Section 2 — Initialize Vertex AI SDK



In [6]:
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION)

## Section 3 — Create Feature Store (Idempotent Pattern)
- Production-grade note: We explicitly check for existence to prevent AlreadyExists errors.

In [7]:
FEATURESTORE_ID = "mlops_featurestore"
# FEATURESTORE_ID = "mlops_featurestore_1"

# List existing feature stores
existing_stores = [fs.name.split("/")[-1] for fs in aiplatform.Featurestore.list()]

if FEATURESTORE_ID in existing_stores:
    featurestore = aiplatform.Featurestore(
        featurestore_name=f"projects/{PROJECT_ID}/locations/{REGION}/featurestores/{FEATURESTORE_ID}"
    )
else:
    featurestore = aiplatform.Featurestore.create(
        featurestore_id=FEATURESTORE_ID,
        online_store_fixed_node_count=1,
        sync=True
    )

print(f"Feature Store ready: {FEATURESTORE_ID}")

Feature Store ready: mlops_featurestore


Architectural note:

- In production, existence checks should be explicit.

- Here we preserve notebook simplicity while remaining re-runnable.

- Explicit existence check avoids AlreadyExists errors.

- sync=True ensures the resource is fully created before proceeding.

## Section 4 — Create Entity Type (Idempotent)
- Production-grade note: Use explicit existence check rather than try/except.


In [8]:
ENTITY_TYPE_ID = "customer"

# List existing entity types
existing_entity_types = [et.name.split("/")[-1] for et in featurestore.list_entity_types()]

if ENTITY_TYPE_ID in existing_entity_types:
    entity_type = aiplatform.EntityType(
        entity_type_name=f"{featurestore.resource_name}/entityTypes/{ENTITY_TYPE_ID}"
    )
else:
    entity_type = featurestore.create_entity_type(
        entity_type_id=ENTITY_TYPE_ID,
        description="Customer entity for streaming demo",
        sync=True
    )

print(f"EntityType ready: {entity_type.resource_name}")

EntityType ready: projects/120008888500/locations/us-central1/featurestores/mlops_featurestore/entityTypes/customer


Notes:

Hardened idempotent pattern prevents duplicate EntityType creation.

In production, entity types are managed via infrastructure-as-code pipelines.

## Section 5 — Define Features (Idempotent)
- Use `batch_create_features` for idempotent feature creation.

In [9]:
feature_configs = [
    {"feature_id": "age", "value_type": "INT64"},
    {"feature_id": "income", "value_type": "DOUBLE"},
    {"feature_id": "churn_risk_score", "value_type": "DOUBLE"},
]

# List existing features
existing_features = [f.name.split("/")[-1] for f in entity_type.list_features()]

# Create only missing features
missing_features = [f for f in feature_configs if f["feature_id"] not in existing_features]

if missing_features:
    # Transform the list of dictionaries into a dictionary format
    # expected by batch_create_features for older SDK versions.
    # The keys will be feature_ids, and values will be dictionaries of feature properties.
    transformed_feature_configs = {
        f["feature_id"]: {"value_type": f["value_type"]} for f in missing_features
    }
    entity_type.batch_create_features(
        feature_configs=transformed_feature_configs,
        sync=True)


**Notes**

- Hardened idempotency ensures no duplicate feature creation.

- sync=True ensures features exist before ingestion.

## Section 5B — Feature Schema

This creates a clear contract between:

- Feature engineering pipeline

- Feature Store

- Model inference

In [10]:
## Section 5B — Feature Schema Documentation

feature_schema = {
    "age": "INT64",
    "income": "DOUBLE",
    "churn_risk_score": "DOUBLE",
}

print("Feature Schema")
print("--------------")

for feature, dtype in feature_schema.items():
    print(f"{feature:20} -> {dtype}")

Feature Schema
--------------
age                  -> INT64
income               -> DOUBLE
churn_risk_score     -> DOUBLE


## Section 6 — Streaming Feature Ingestion
- Production-grade note: Use supported `write_feature_values` via Gapic client.
- Single-entity ingestion demonstration.

In [11]:
from google.cloud.aiplatform_v1.types import FeatureValue

# Feature schema
# age -> INT64
# income -> DOUBLE
# churn_risk_score -> DOUBLE

write_payloads = [{
    "entity_id": "customer_001",
    "feature_values": {
        "age": FeatureValue(int64_value=42),
        "income": FeatureValue(double_value=85000.0),
        "churn_risk_score": FeatureValue(double_value=0.17),
    }
}]

response = entity_type.write_feature_values(write_payloads)

print("Streaming ingestion complete:", response)

Streaming ingestion complete: <google.cloud.aiplatform.featurestore.entity_type.EntityType object at 0x7e4c696fb680> 
resource name: projects/120008888500/locations/us-central1/featurestores/mlops_featurestore/entityTypes/customer


Notes:

Use timezone-aware datetimes for feature timestamps.

This demonstrates real-time streaming ingestion without batch jobs.

Optional: In production, ingestion should be handled via automated pipelines, not notebooks.

This demonstrates:

-   Direct streaming

-   No batch import

-   Real-time feature update

-   SDK-level ingestion (as course requires)

## Section 7 — Online Feature Retrieval

In [13]:
online_features = entity_type.read(["customer_001"])

print("Online Features retrieved successfully:")
print(online_features)

Online Features retrieved successfully:
      entity_id  churn_risk_score  age   income
0  customer_001              0.17   42  85000.0


## Section 7B - Infrastructure Summary

In [14]:
## Section 7B — Infrastructure Summary

print("Vertex AI Feature Store Infrastructure")
print("---------------------------------------")

print(f"Project: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Feature Store: {FEATURESTORE_ID}")
print(f"Entity Type: {ENTITY_TYPE_ID}")

print("\nFeatures Registered")

features = entity_type.list_features()

for feature in features:
    print("-", feature.name.split("/")[-1])

print("\nOnline Serving Status: ENABLED")
print("Streaming Ingestion: SUCCESS")
print("Online Retrieval: SUCCESS")

Vertex AI Feature Store Infrastructure
---------------------------------------
Project: first-haven-485022-n5
Region: us-central1
Feature Store: mlops_featurestore
Entity Type: customer

Features Registered
- age
- churn_risk_score
- income

Online Serving Status: ENABLED
Streaming Ingestion: SUCCESS
Online Retrieval: SUCCESS


## 7C - Simulated Model Feature Lookup

This section simulates how a deployed model retrieves features from the online Feature Store during inference.

In production systems, models do not compute features at prediction time. Instead, they retrieve the latest feature values from the online store to ensure training/serving consistency.



In [15]:
# Simulated model feature lookup

entity_id = "customer_001"

features = entity_type.read(
    entity_ids=[entity_id]
)

# features = entity_type.read_feature_values(
#     entity_ids=[entity_id],
#     feature_ids=["age", "income", "churn_risk_score"]
# )

print("Retrieved feature values for inference:")
print(features)

Retrieved feature values for inference:
      entity_id  churn_risk_score  age   income
0  customer_001              0.17   42  85000.0


## Section 8 — Cleanup

In [16]:
featurestore.delete(force=True, sync=True)

print("Featurestore deleted.")

Featurestore deleted.


## Production Notes / Best Practices
- **Idempotency**: Always check existence before creation.
- **Secrets**: Use environment-driven configuration, no hardcoding.
- **Separation of concerns**: Feature pipelines should populate Feature Store, not notebooks.
- **Cost awareness**: Online node count directly affects billing.
- **Exam relevance**: Know differences between batch vs streaming ingestion, offline vs online store, and feature vs model monitoring.


**Idempotency**:
- FeatureStore, EntityType, and Features use explicit existence checks
instead of try/except.
- Use infrastructure-as-code for feature definitions.

**Separation of Concerns**:
- Feature pipelines should populate Feature Store; notebooks are demonstration-only.

**Cost Awareness**:
- Online store node count affects billing.

**Exam Relevance**:
- Know the difference between batch vs streaming ingestion
- Offline vs online store
- Feature monitoring vs model monitoring

    **Monitoring**:
    - Consider drift monitoring and data validation for production pipelines.

## Section 9 Freeze Working Environment

In [17]:
!pip show google-cloud-aiplatform

Name: google-cloud-aiplatform
Version: 1.140.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, google-genai, packaging, proto-plus, protobuf, pydantic, typing_extensions
Required-by: google-adk


In [18]:
import google.cloud.aiplatform as aip
print(aip.__version__)

1.140.0


In [19]:
!pip freeze > requirements.txt